# Notebook 03 — Data Engineering and Final Dataset

## 1. Notebook Purpose

This notebook cleans the raw datasets checked in
`notebooks/01_dataset_sources_and_description.ipynb` and explored in
`notebooks/02_eda_dataset_understanding.ipynb`, and builds the final
research-ready dataset used by the econometric and machine learning notebooks
(04–06).

**Boundaries (see `instructions.md` §7)**
- Cleaning, variable construction, merging, and interaction variables only.
- **No** econometric models and **no** machine learning models in this notebook.
- No fake data and no simulated results. If a required raw file is missing, this
  notebook stops with a clear message.
- Every recode uses the ESS variable mapping established in notebook 02. All codes
  have been confirmed against the ESS11 codebook (`data/raw/ess/ESS11_codebook.pdf`).

**Notebook structure**
1. Notebook purpose
2. Load raw datasets
3. Clean ESS individual-level data
4. Create dependent variable
5. Create independent variables
6. Clean country-level AI adoption data
7. Merge country-level data with ESS
8. Create interaction variables
9. Final missing value and quality checks
10. Save final dataset
11. Save variable dictionary and data engineering report

**Main outputs**
- `data/processed/final_employment_ai_dataset.csv`
- `data/pickle/final_employment_ai_dataset.pkl`
- `docs/variable_dictionary.md`
- `outputs/reports/data_engineering_report.md`
- `outputs/tables/final_dataset_summary.xlsx`

This notebook is designed to run once per data update and save all outputs; it does
not need to be rerun unless the raw data or a variable definition changes.

## Environment Setup

Run the install cell below **once** if you get a `ModuleNotFoundError` (for
example, `No module named 'pandas'`) — skip it if packages are already installed.
Make sure the correct kernel/virtual environment is selected in VS Code before
running any other cell. Full setup instructions:
[docs/environment_setup.md](../docs/environment_setup.md).

In [1]:
# OPTIONAL: run only if a package is missing (e.g. ModuleNotFoundError: No module named 'pandas').
# Safe to skip if all packages in requirements.txt are already installed in this kernel.
from pathlib import Path

_project_root_for_install = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
_requirements_path = _project_root_for_install / 'requirements.txt'

if not _requirements_path.exists():
    print(f'requirements.txt not found at {_requirements_path}. See docs/environment_setup.md.')
else:
    %pip install -r "{_requirements_path}"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Imports and path setup for this notebook (data engineering: pandas, numpy, pyreadstat).
try:
    import numpy as np
    import pandas as pd
    import pyreadstat
except ImportError as exc:
    raise ImportError(
        f"Missing package: {exc.name}. Run the optional install cell above (or "
        "`pip install -r requirements.txt` in a terminal), then restart the kernel and re-run this cell."
    ) from exc

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
PICKLE_DIR = DATA_DIR / 'pickle'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
TABLES_DIR = OUTPUTS_DIR / 'tables'
REPORTS_DIR = OUTPUTS_DIR / 'reports'
MODELS_DIR = OUTPUTS_DIR / 'models'
DOCS_DIR = PROJECT_ROOT / 'docs'
for d in [PROCESSED_DIR, PICKLE_DIR, FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

cleaning_log = []  # list of (step, detail) strings, written to the data engineering report at the end

def log_step(step, detail):
    cleaning_log.append((step, detail))
    print(f'{step}: {detail}')

print('Project root:', PROJECT_ROOT)

Project root: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis


## 2. Load Raw Datasets

ESS individual-level data is required — the notebook stops here if it is missing.
Eurostat AI adoption data and ESS Multilevel data are used if present; their
absence is documented but does not stop the notebook, since they affect only
specific variables/sections further down.

In [3]:
# ESS11 is the round used for all analysis in this thesis. The loader looks ONLY
# for ESS11 (CSV preferred, SPSS .sav accepted) and stops with a clear message if
# it is missing -- it deliberately does NOT fall back to any other round (e.g.
# ESS10), so results can never be silently produced from the wrong dataset.
# See data/raw/ess/README.md for the exact ESS11 download steps and file location.
ess_candidates = [
    RAW_DIR / 'ess' / 'ESS11.csv',
    RAW_DIR / 'ess' / 'ESS11.sav',
]
ess_path = next((p for p in ess_candidates if p.exists()), None)
if ess_path is None:
    raise FileNotFoundError(
        'Required ESS11 raw file not found in data/raw/ess/. Expected one of: '
        f"{[p.name for p in ess_candidates]}. "
        'Follow the download steps in data/raw/ess/README.md (ESS cannot be '
        'redistributed, so the file must be downloaded manually). '
        'Do not proceed with simulated data or a different ESS round.'
    )

if ess_path.suffix == '.csv':
    df_ess_raw = pd.read_csv(ess_path, low_memory=False)
    meta_ess = None  # SPSS variable/value labels are only available when loading a .sav file
else:
    df_ess_raw, meta_ess = pyreadstat.read_sav(ess_path)

log_step('load_ess', f'Loaded {ess_path.name} with shape {df_ess_raw.shape}')

load_ess: Loaded ESS11.csv with shape (50116, 691)


In [4]:
eurostat_files = list((RAW_DIR / 'eurostat_ai').glob('*.csv'))
if not eurostat_files:
    print('No Eurostat AI adoption file found in data/raw/eurostat_ai/.')
    print('Run notebooks/01_dataset_sources_and_description.ipynb to check/retrieve it. '
          'ai_adoption_enterprises_pct and its interaction terms will be all-missing until it is available.')
    df_ai_raw = None
else:
    df_ai_raw = pd.read_csv(eurostat_files[0])
    log_step('load_eurostat_ai', f'Loaded {eurostat_files[0].name} with shape {df_ai_raw.shape}')

load_eurostat_ai: Loaded isoc_eb_ai.csv with shape (620, 10)


In [5]:
# ESS Multilevel data is loaded COLUMN-SELECTIVELY: the raw file has ~9,300
# respondent-level columns, and loading it in full (or even calling .copy() on the
# full frame) previously caused a MemoryError. We first read only the header (no
# data rows) to find the country key and the macro-indicator columns matching
# MACRO_KEYWORDS, then re-read the file with only those columns via `usecols`.
# MACRO_KEYWORDS is reused as-is in section 7, where these same matched columns
# are aggregated to one row per country and merged.
MACRO_KEYWORDS = ['unempl', 'gdp', 'gini', 'socx', 'migr', 'educ']

ess_multilevel_candidates = [
    RAW_DIR / 'ess_multilevel' / 'ESS_multilevel_data.csv',
    RAW_DIR / 'ess_multilevel' / 'ESS_multilevel_data.sav',
]
ess_multilevel_path = next((p for p in ess_multilevel_candidates if p.exists()), None)

df_macro_raw = None
macro_country_col = None
matched_macro_cols = []

if ess_multilevel_path is None:
    print('ESS Multilevel file not found in data/raw/ess_multilevel/.')
    print('Only ai_adoption_enterprises_pct will be available as a country-level variable; no macro controls will be added.')
else:
    try:
        if ess_multilevel_path.suffix == '.csv':
            header_cols = list(pd.read_csv(ess_multilevel_path, nrows=0).columns)
        else:
            header_cols = list(pyreadstat.read_sav(ess_multilevel_path, metadataonly=True)[0].columns)

        header_cols_lower = {c: c.strip().lower() for c in header_cols}
        macro_country_col = next((c for c, lc in header_cols_lower.items() if lc in ('cntry', 'country')), None)
        matched_macro_cols = [
            c for c, lc in header_cols_lower.items()
            if c != macro_country_col and any(k in lc for k in MACRO_KEYWORDS)
        ]

        if macro_country_col is None:
            print('Could not identify the country key ("cntry"/"country") in the ESS Multilevel file header; macro controls skipped.')
        elif not matched_macro_cols:
            print(f'No columns in the ESS Multilevel header matched the expected keywords {MACRO_KEYWORDS}; macro controls skipped.')
        else:
            usecols = [macro_country_col] + matched_macro_cols
            if ess_multilevel_path.suffix == '.csv':
                df_macro_raw = pd.read_csv(ess_multilevel_path, usecols=usecols, low_memory=False)
            else:
                df_macro_raw, _ = pyreadstat.read_sav(ess_multilevel_path, usecols=usecols)
            log_step(
                'load_ess_multilevel',
                f'Loaded {ess_multilevel_path.name} column-selectively: read {len(usecols)} of {len(header_cols)} '
                f'columns (country key + {len(matched_macro_cols)} macro-keyword columns), avoiding the full '
                f'respondent-level load (and the .copy() of it) that previously caused a MemoryError.',
            )
    except Exception as exc:
        df_macro_raw = None
        print(
            f'Could not selectively load the ESS Multilevel file ({exc}); macro controls skipped. '
            'The final dataset will still include ai_adoption_enterprises_pct.'
        )

load_ess_multilevel: Loaded ESS_multilevel_data.csv column-selectively: read 1268 of 9348 columns (country key + 1267 macro-keyword columns), avoiding the full respondent-level load (and the .copy() of it) that previously caused a MemoryError.


## 3. Clean ESS Individual-Level Data

Standardizes column names, defines the ESS special missing-value codes per
variable, and converts them to `NaN` rather than treating them as valid numeric
values. Original raw columns are kept alongside the cleaned ones during
processing (only the final selected variables are kept in the saved dataset in
section 10).

In [6]:
df = df_ess_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]
log_step('standardize_columns', 'Lower-cased and stripped all ESS column names')

standardize_columns: Lower-cased and stripped all ESS column names


In [7]:
# ESS variable mapping (see notebooks/02_eda_dataset_understanding.ipynb, section 4).
# Codes below were confirmed against the ESS11 codebook
# (data/raw/ess/ESS11_codebook.pdf) before building the final dataset.
COL_EMPLOYMENT_STATUS = 'mnactic'  # main activity, POST-CODED for ALL respondents
COL_EMPLOYED_CODE = 1              # 1 = paid work; 2..9 = not in paid work
COL_GENDER = 'gndr'
COL_GENDER_FEMALE_CODE = 2
COL_AGE = 'agea'
COL_EDUCATION_YEARS = 'eduyrs'
COL_EDUCATION_LEVEL = 'eisced'
COL_HIGH_EDUCATION_THRESHOLD = 5  # ISCED level treated as tertiary or above
COL_MARITAL = 'maritalb'
COL_MARRIED_CODE = 1
COL_CHILDREN_HOME = 'chldhm'  # not fielded in ESS11; the household grid is used instead (section 5)
COL_HOUSEHOLD_SIZE = 'hhmmb'
COL_MIGRATION = 'brncntr'
COL_MIGRATION_CODE = 2  # 'not born in country'
COL_HEALTH = 'health'
COL_DIGITAL = 'netusoft'
COL_TRUST_ITEMS = ['trstprl', 'trstlgl', 'trstplc', 'trstplt']
COL_DISCRIMINATION = 'dscrgrp'
COL_DISCRIMINATION_CODE = 1
COL_COUNTRY = 'cntry'
COL_WEIGHT = 'anweight'

# Household grid (used to build a real 'child under 16 in household' indicator in
# section 5): birth years of co-resident members 2..13, plus the interview date
# used to turn those birth years into ages at the time of interview.
COL_INTERVIEW_DATE = 'inwds'          # interview start timestamp (year is used)
GRID_SLOTS = range(2, 14)             # ESS household grid members 2..13
COL_GRID_YRBRN = 'yrbrn'              # yrbrn2..yrbrn13 birth years
YRBRN_MISSING_CODES = {6666, 7777, 8888, 9999}
CHILD_AGE_CUTOFF = 16                 # 'child' = co-resident member aged under 16

# ESS special missing-value codes by variable, confirmed against the ESS11 codebook.
MISSING_CODES = {
    COL_EMPLOYMENT_STATUS: [77, 88, 99],
    COL_GENDER: [9],
    COL_AGE: [999],
    COL_EDUCATION_YEARS: [77, 88, 99],
    COL_EDUCATION_LEVEL: [55, 77, 88, 99],
    COL_MARITAL: [66, 77, 88, 99],
    COL_CHILDREN_HOME: [7, 8, 9],
    COL_HOUSEHOLD_SIZE: [77, 88, 99],
    COL_MIGRATION: [7, 8, 9],
    COL_HEALTH: [7, 8, 9],
    COL_DIGITAL: [7, 8, 9],
    COL_DISCRIMINATION: [7, 8, 9],
    **{col: [77, 88, 99] for col in COL_TRUST_ITEMS},
}

def clean_special_missing(frame, column, codes):
    """Replaces ESS special missing-value codes with NaN for one column, in place.
    Returns the number of values converted, for the cleaning log."""
    if column not in frame.columns:
        return None
    mask = frame[column].isin(codes)
    n_converted = int(mask.sum())
    frame.loc[mask, column] = np.nan
    return n_converted

In [8]:
missing_conversion_summary = {}
for column, codes in MISSING_CODES.items():
    n_converted = clean_special_missing(df, column, codes)
    if n_converted is None:
        print(f'Column not found, skipped: {column}')
    else:
        missing_conversion_summary[column] = n_converted

log_step(
    'special_missing_recode',
    f'Converted ESS special missing codes to NaN for {len(missing_conversion_summary)} columns: {missing_conversion_summary}',
)

Column not found, skipped: chldhm
special_missing_recode: Converted ESS special missing codes to NaN for 15 columns: {'mnactic': 279, 'gndr': 0, 'agea': 393, 'eduyrs': 901, 'eisced': 382, 'maritalb': 811, 'hhmmb': 337, 'brncntr': 32, 'health': 76, 'netusoft': 50, 'dscrgrp': 514, 'trstprl': 998, 'trstlgl': 1149, 'trstplc': 575, 'trstplt': 779}


## 4. Create Dependent Variable

**Variable chosen**: `mnactic` (main activity, last 7 days — *post-coded*),
coded **1 = paid work**, with every other category (2–9: education, unemployment,
retirement, permanent sickness/disability, community/military service, housework,
other) coded 0.

**Why `mnactic` and not `mainact`.** In the ESS questionnaire respondents first
tick *every* activity that applied in the last seven days (item F17a). `mainact`
is only asked as a follow-up of the minority who ticked more than one activity
("ASK IF MORE THAN ONE CODED AT F17a"); everyone with a single clear activity is
left as code 66, *not applicable*. Building employment from `mainact` therefore
keeps only a small, self-selected subsample (people whose situation was ambiguous
enough to report several activities) and discards ~87% of respondents on a
non-random basis. `mnactic` is the post-coded main-activity variable available for
**all respondents** ("All respondents. Post coded" in the codebook), so it is the
correct item for analysis. Confirmed against `data/raw/ess/ESS11_codebook.pdf`.

In [9]:
if COL_EMPLOYMENT_STATUS not in df.columns:
    raise KeyError(
        f'{COL_EMPLOYMENT_STATUS} not found in the ESS data. Update the mapping in '
        'section 3 against the codebook before continuing.'
    )

df['employed'] = np.where(df[COL_EMPLOYMENT_STATUS].isna(), np.nan, np.where(df[COL_EMPLOYMENT_STATUS] == COL_EMPLOYED_CODE, 1, 0))

n_missing_employed = df['employed'].isna().sum()
n_before_drop = len(df)
df = df.dropna(subset=['employed']).copy()
df['employed'] = df['employed'].astype(int)
n_after_drop = len(df)

log_step(
    'dependent_variable',
    f"Built 'employed' from {COL_EMPLOYMENT_STATUS} (code {COL_EMPLOYED_CODE} = employed). "
    f'Dropped {n_before_drop - n_after_drop} of {n_before_drop} rows with missing employment status '
    f'(reason: dependent variable cannot be classified). Remaining distribution: {df["employed"].value_counts(normalize=True).to_dict()}',
)

dependent_variable: Built 'employed' from mnactic (code 1 = employed). Dropped 279 of 50116 rows with missing employment status (reason: dependent variable cannot be classified). Remaining distribution: {1: 0.5152196159479905, 0: 0.48478038405200957}


## 5. Create Independent Variables

Builds every core individual-level variable. Each recode is documented inline;
the same documentation is written to `docs/variable_dictionary.md` in section 11.

In [10]:
df['age'] = df[COL_AGE]
df['age_squared'] = df['age'] ** 2
df['female'] = np.where(df[COL_GENDER] == COL_GENDER_FEMALE_CODE, 1, 0)
df['education_years'] = df[COL_EDUCATION_YEARS]
df['education_level'] = df[COL_EDUCATION_LEVEL]
df['high_education'] = np.where(df['education_level'] >= COL_HIGH_EDUCATION_THRESHOLD, 1, 0)
df['married'] = np.where(df[COL_MARITAL] == COL_MARRIED_CODE, 1, 0)
df['household_size'] = df[COL_HOUSEHOLD_SIZE]
df['migration_background'] = np.where(df[COL_MIGRATION] == COL_MIGRATION_CODE, 1, 0)
df['health_status'] = df[COL_HEALTH]
df['country'] = df[COL_COUNTRY]

log_step('core_variables', 'Built age, age_squared, female, education_years, education_level, high_education, married, household_size, migration_background, health_status, country')

core_variables: Built age, age_squared, female, education_years, education_level, high_education, married, household_size, migration_background, health_status, country


### `children_household` — child under 16 in the household

Built from the ESS household grid rather than a household-size proxy. `chldhm`
is not fielded in ESS11, but the grid records the birth year (`yrbrn2`–`yrbrn13`)
of each co-resident household member. A respondent is coded **1** if **any**
co-resident member has an implied age under 16 at the time of interview
(interview year, taken from `inwds`, minus the member's birth year) and **0**
otherwise. This is a direct measure of the presence of a dependent child in the
household, replacing the earlier `household_size > 1` proxy, and brings H3 closer
to the childcare mechanism described in Chapter I.

In [11]:
# Interview year (from the interview start timestamp), used to convert household
# members' birth years into ages. Rows with an unparseable date fall back to the
# sample's median interview year so a valid child flag can still be computed.
interview_year = pd.to_datetime(df[COL_INTERVIEW_DATE], errors='coerce').dt.year
interview_year = interview_year.fillna(interview_year.median())

grid_year_cols = [f'{COL_GRID_YRBRN}{k}' for k in GRID_SLOTS if f'{COL_GRID_YRBRN}{k}' in df.columns]

if grid_year_cols:
    child_flags = []
    for ycol in grid_year_cols:
        byear = pd.to_numeric(df[ycol], errors='coerce')
        byear = byear.where(~byear.isin(YRBRN_MISSING_CODES))
        member_age = interview_year - byear
        child_flags.append(((member_age >= 0) & (member_age < CHILD_AGE_CUTOFF)).fillna(False))
    df['children_household'] = pd.concat(child_flags, axis=1).any(axis=1).astype(int)
    n_child = int(df['children_household'].sum())
    log_step(
        'children_household',
        f'Built real "child under {CHILD_AGE_CUTOFF} in household" indicator from the ESS '
        f'household grid ({len(grid_year_cols)} birth-year columns + interview year from '
        f'{COL_INTERVIEW_DATE}): {n_child} of {len(df)} respondents '
        f'({n_child / len(df):.1%}) have at least one co-resident child under {CHILD_AGE_CUTOFF}.',
    )
else:
    df['children_household'] = np.where(df['household_size'].fillna(0) > 1, 1, 0)
    log_step(
        'children_household',
        'Household grid (yrbrn2..) not found in this file; fell back to the '
        'household_size > 1 proxy. This is not expected for ESS11 and should be checked.',
    )

children_household: Built real "child under 16 in household" indicator from the ESS household grid (12 birth-year columns + interview year from inwds): 10909 of 49837 respondents (21.9%) have at least one co-resident child under 16.


### `digital_readiness`

**Individual-level proxy only.** ESS does not measure individual AI usage, so this
variable uses `netusoft` (internet use frequency) as the best available proxy for
digital readiness, ordered so that higher values indicate more frequent internet
use. AI adoption itself is only ever used as the country-level variable
`ai_adoption_enterprises_pct` (section 6) — individual-level AI usage is not
invented or simulated.

In [12]:
df['digital_readiness'] = df[COL_DIGITAL]
log_step('digital_readiness', f'Built from {COL_DIGITAL} (internet use frequency), used as an individual-level proxy only')

digital_readiness: Built from netusoft (internet use frequency), used as an individual-level proxy only


### `institutional_trust_index` and `discrimination_experience`

In [13]:
available_trust_items = [c for c in COL_TRUST_ITEMS if c in df.columns]
if available_trust_items:
    df['institutional_trust_index'] = df[available_trust_items].mean(axis=1, skipna=True)
    log_step('institutional_trust_index', f'Built as the row-wise mean of available trust items: {available_trust_items}')
else:
    df['institutional_trust_index'] = np.nan
    log_step('institutional_trust_index', 'No trust items found in this file; variable is all-missing until the mapping is corrected')

if COL_DISCRIMINATION in df.columns:
    df['discrimination_experience'] = np.where(df[COL_DISCRIMINATION] == COL_DISCRIMINATION_CODE, 1, 0)
    log_step('discrimination_experience', f'Built from {COL_DISCRIMINATION}')
else:
    df['discrimination_experience'] = np.nan
    log_step('discrimination_experience', f'{COL_DISCRIMINATION} not found; variable is all-missing until the mapping is corrected')

if COL_WEIGHT in df.columns:
    df['survey_weight'] = df[COL_WEIGHT]
    log_step('survey_weight', f'Copied from {COL_WEIGHT}')
else:
    df['survey_weight'] = np.nan
    log_step('survey_weight', f'{COL_WEIGHT} not found; weighted models in notebook 05 will note that weighting was not available')

institutional_trust_index: Built as the row-wise mean of available trust items: ['trstprl', 'trstlgl', 'trstplc', 'trstplt']
discrimination_experience: Built from dscrgrp
survey_weight: Copied from anweight


## 6. Clean Country-Level AI Adoption Data

Builds `ai_adoption_enterprises_pct`: the percentage of enterprises using at
least one AI technology, by country. If Eurostat publishes multiple years, the
latest available year is used and documented below rather than averaging across
years.

In [14]:
if df_ai_raw is None:
    df_ai = pd.DataFrame(columns=['country', 'ai_adoption_enterprises_pct'])
    ai_year_used = None
    eurostat_rows_before_filtering = 0
    eurostat_rows_after_filtering = 0
    eurostat_unique_countries = 0
    log_step('clean_eurostat_ai', 'No raw Eurostat AI file available; ai_adoption_enterprises_pct will be all-missing after merge')
else:
    eurostat_rows_before_filtering = len(df_ai_raw)
    geo_col_candidates = ['geo\\TIME_PERIOD', 'geo']
    geo_col = next((c for c in geo_col_candidates if c in df_ai_raw.columns), None)
    if geo_col is None:
        raise KeyError('[EUROSTAT FILTER NEEDS MANUAL CHECK] Country/geo column was not found.')

    required_dimension_values = {
        'freq': 'A',
        'size_emp': 'GE10',
        'nace_r2': 'C10-S951_X_K',
        'indic_is': 'E_AI_TANY',
        'unit': 'PC_ENT',
    }
    missing_dimensions = [c for c in required_dimension_values if c not in df_ai_raw.columns]
    if missing_dimensions:
        raise KeyError(
            '[EUROSTAT FILTER NEEDS MANUAL CHECK] Missing required dimensions: '
            f'{missing_dimensions}'
        )

    if '2025' not in df_ai_raw.columns:
        raise KeyError('[EUROSTAT FILTER NEEDS MANUAL CHECK] Year 2025 is not available.')
    ai_year_used = '2025'

    filter_mask = pd.Series(True, index=df_ai_raw.index)
    for dimension, required_value in required_dimension_values.items():
        filter_mask &= df_ai_raw[dimension].eq(required_value)

    df_ai_filtered = df_ai_raw.loc[
        filter_mask,
        [geo_col, ai_year_used],
    ].copy()
    df_ai_filtered[ai_year_used] = pd.to_numeric(
        df_ai_filtered[ai_year_used], errors='coerce'
    )
    df_ai_filtered = df_ai_filtered.dropna(subset=[geo_col, ai_year_used])

    # Eurostat uses EL for Greece, while ESS uses the ISO code GR.
    df_ai_filtered[geo_col] = df_ai_filtered[geo_col].replace({'EL': 'GR'})

    # Keep only geographic codes present in the cleaned ESS sample. This removes
    # Eurostat aggregates such as EU27_2020 and EA before the country-level merge.
    ess_country_codes = set(df['country'].dropna().unique())
    df_ai_filtered = df_ai_filtered[df_ai_filtered[geo_col].isin(ess_country_codes)]

    df_ai = df_ai_filtered.rename(
        columns={geo_col: 'country', ai_year_used: 'ai_adoption_enterprises_pct'}
    )
    eurostat_rows_after_filtering = len(df_ai)
    eurostat_unique_countries = df_ai['country'].nunique()

    duplicate_countries = sorted(
        df_ai.loc[df_ai['country'].duplicated(keep=False), 'country'].unique()
    )
    if duplicate_countries:
        raise ValueError(
            '[EUROSTAT FILTER NEEDS MANUAL CHECK] Country is not unique after filtering: '
            f'{duplicate_countries}'
        )
    if eurostat_rows_after_filtering != eurostat_unique_countries:
        raise AssertionError('Filtered Eurostat row and unique-country counts differ.')

    log_step(
        'clean_eurostat_ai',
        'Filtered isoc_eb_ai to freq=A, size_emp=GE10, '
        'nace_r2=C10-S951_X_K, indic_is=E_AI_TANY '
        '(enterprises using at least one AI technology), unit=PC_ENT, year=2025, '
        'and ESS country codes. '
        f'Rows before filtering: {eurostat_rows_before_filtering}; rows after filtering: '
        f'{eurostat_rows_after_filtering}; unique countries: {eurostat_unique_countries}.',
    )

df_ai

clean_eurostat_ai: Filtered isoc_eb_ai to freq=A, size_emp=GE10, nace_r2=C10-S951_X_K, indic_is=E_AI_TANY (enterprises using at least one AI technology), unit=PC_ENT, year=2025, and ESS country codes. Rows before filtering: 620; rows after filtering: 25; unique countries: 25.


,country,ai_adoption_enterprises_pct
461,AT,29.95
463,BE,34.54
464,BG,8.55
465,CY,9.27
467,DE,25.97
470,EE,23.40
471,GR,8.93
472,ES,20.27
474,FI,37.82
475,FR,18.16


## 7. Merge Country-Level Data with ESS

Merges the cleaned AI adoption data, and — if available — ESS Multilevel/macro
indicators, onto the individual-level ESS data using the country code as the
merge key. Merge quality (countries before/after, unmatched countries, missing
values introduced) is checked and reported explicitly rather than assumed.

In [15]:
ess_countries_before = set(df['country'].dropna().unique())
n_countries_before = len(ess_countries_before)
n_rows_before_ai_merge = len(df)

df = df.merge(
    df_ai,
    on='country',
    how='left',
    validate='many_to_one',
)
n_rows_after_ai_merge = len(df)
if n_rows_after_ai_merge != n_rows_before_ai_merge:
    raise AssertionError(
        'Eurostat many-to-one merge changed the ESS row count: '
        f'{n_rows_before_ai_merge} -> {n_rows_after_ai_merge}'
    )

ai_countries = set(df_ai['country'].unique()) if not df_ai.empty else set()
unmatched_countries = sorted(ess_countries_before - ai_countries)
matched_countries = sorted(ess_countries_before & ai_countries)
n_missing_ai = int(df['ai_adoption_enterprises_pct'].isna().sum())

merge_report_ai = {
    'rows_before_merge': n_rows_before_ai_merge,
    'rows_after_merge': n_rows_after_ai_merge,
    'countries_before_merge': n_countries_before,
    'countries_matched_to_ai_data': len(matched_countries),
    'unmatched_countries': unmatched_countries,
    'rows_with_missing_ai_adoption': n_missing_ai,
    'ai_data_year_used': ai_year_used,
    'merge_validation': 'many_to_one',
}
log_step('merge_ai_adoption', str(merge_report_ai))
merge_report_ai

merge_ai_adoption: {'rows_before_merge': 49837, 'rows_after_merge': 49837, 'countries_before_merge': 30, 'countries_matched_to_ai_data': 25, 'unmatched_countries': ['CH', 'GB', 'IL', 'IS', 'UA'], 'rows_with_missing_ai_adoption': 7419, 'ai_data_year_used': '2025', 'merge_validation': 'many_to_one'}

{'rows_before_merge': 49837,
 'rows_after_merge': 49837,
 'countries_before_merge': 30,
 'countries_matched_to_ai_data': 25,
 'unmatched_countries': ['CH', 'GB', 'IL', 'IS', 'UA'],
 'rows_with_missing_ai_adoption': 7419,
 'ai_data_year_used': '2025',
 'merge_validation': 'many_to_one'}

In [16]:
# Macro/institutional controls from ESS Multilevel data. Only columns that are
# clearly present and clearly interpretable are added. The respondent-level source
# is aggregated to one row per country before the protected many-to-one merge.
n_rows_before_macro_merge = len(df)

if df_macro_raw is None:
    log_step(
        'merge_macro',
        'ESS Multilevel data not available or macro columns could not be identified '
        'in the header; no macro controls added beyond ai_adoption_enterprises_pct',
    )
else:
    # Exclude columns that would collide with existing ESS variables.
    colliding_macro_cols = [c for c in matched_macro_cols if c in df.columns]
    candidate_macro_cols = [c for c in matched_macro_cols if c not in colliding_macro_cols]

    # Country-level indicators must be numeric to be aggregated by mean.
    for c in candidate_macro_cols:
        df_macro_raw[c] = pd.to_numeric(df_macro_raw[c], errors='coerce')
    numeric_macro_cols = [c for c in candidate_macro_cols if df_macro_raw[c].notna().any()]
    dropped_non_numeric_cols = [c for c in candidate_macro_cols if c not in numeric_macro_cols]

    if not numeric_macro_cols:
        log_step(
            'merge_macro',
            'No usable numeric country-level columns found among the keyword matches '
            f'(after excluding {len(colliding_macro_cols)} colliding columns and '
            f'{len(dropped_non_numeric_cols)} non-numeric columns); no macro controls added',
        )
    else:
        macro_subset = (
            df_macro_raw[[macro_country_col] + numeric_macro_cols]
            .groupby(macro_country_col, as_index=False)
            .mean(numeric_only=True)
            .rename(columns={macro_country_col: 'country'})
            .drop_duplicates(subset='country')
        )

        n_countries_before_macro = df['country'].nunique()
        try:
            df = df.merge(
                macro_subset,
                on='country',
                how='left',
                validate='many_to_one',
            )
        except pd.errors.MergeError as exc:
            log_step(
                'merge_macro',
                'Skipped macro merge because country was not unique in the aggregated '
                f'ESS Multilevel data ({exc}).',
            )
        else:
            n_missing_macro = {c: int(df[c].isna().sum()) for c in numeric_macro_cols}
            note_parts = []
            if colliding_macro_cols:
                note_parts.append(
                    f'excluded {len(colliding_macro_cols)} columns colliding with existing '
                    f'ESS variable names (e.g. {colliding_macro_cols[:5]})'
                )
            if dropped_non_numeric_cols:
                note_parts.append(
                    f'dropped {len(dropped_non_numeric_cols)} non-numeric matched columns'
                )
            note = f' ({"; ".join(note_parts)}.)' if note_parts else ''
            log_step(
                'merge_macro',
                f'Added {len(numeric_macro_cols)} macro columns from ESS Multilevel data, '
                f'aggregated to one row per country before a many-to-one merge.{note} '
                f'Missing values per column after merge: {n_missing_macro}.',
            )

n_rows_after_macro_merge = len(df)
if n_rows_after_macro_merge != n_rows_before_macro_merge:
    raise AssertionError(
        'ESS Multilevel many-to-one merge changed the ESS row count: '
        f'{n_rows_before_macro_merge} -> {n_rows_after_macro_merge}'
    )
log_step(
    'country_merge_row_check',
    f'Rows before Eurostat merge: {n_rows_before_ai_merge}; '
    f'after Eurostat merge: {n_rows_after_ai_merge}; '
    f'after ESS Multilevel merge: {n_rows_after_macro_merge}. '
    'No duplicate respondent expansion from country-level merges.',
)

C:\Users\khamidov.m\AppData\Local\Temp\ipykernel_28176\1482661204.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .mean(numeric_only=True)


merge_macro: Added 1265 macro columns from ESS Multilevel data, aggregated to one row per country before a many-to-one merge. (excluded 2 columns colliding with existing ESS variable names (e.g. ['educde2', 'educgb1']).) Missing values per column after merge: {'c_cnmigrat_2000': 4824, 'c_cnmigrat_2001': 4824, 'c_cnmigrat_2002': 4824, 'c_cnmigrat_2003': 4824, 'c_cnmigrat_2004': 4824, 'c_cnmigrat_2005': 4824, 'c_cnmigrat_2006': 4824, 'c_cnmigrat_2007': 4824, 'c_cnmigrat_2008': 4824, 'c_cnmigrat_2009': 4824, 'c_cnmigrat_2010': 4824, 'c_cnmigrat_2011': 4824, 'c_cnmigrat_2012': 4824, 'c_cnmigrat_2013': 4824, 'c_cnmigrat_2014': 4824, 'c_cnmigrat_2015': 4824, 'c_cnmigrat_2016': 4824, 'c_cnmigrat_2017': 4824, 'c_cnmigrat_2018': 4824, 'c_cnmigrat_2019': 6492, 'c_cnmigrat_2020': 6492, 'c_cnmigrat_2021': 6492, 'c_cnmigrat_2022': 6492, 'c_cnmigrat_2023': 6492, 'c_cnmigratrt_2000': 4824, 'c_cnmigratrt_2001': 4824, 'c_cnmigratrt_2002': 4824, 'c_cnmigratrt_2003': 4824, 'c_cnmigratrt_2004': 4824, 'c_c

## 8. Create Interaction Variables

Only the interaction variables required by the thesis hypotheses are created (see
`docs/variable_dictionary.md` and `docs/methodology_notes.md`).

In [17]:
df['education_years_x_ai_adoption'] = df['education_years'] * df['ai_adoption_enterprises_pct']  # H2
df['digital_readiness_x_ai_adoption'] = df['digital_readiness'] * df['ai_adoption_enterprises_pct']  # H2
df['female_x_children'] = df['female'] * df['children_household']  # H3
df['migration_x_education'] = df['migration_background'] * df['education_years']  # H4
df['migration_x_digital_readiness'] = df['migration_background'] * df['digital_readiness']  # H4

# Optional triple interaction: only kept if all three components are usable
# (i.e. AI adoption was actually merged in); otherwise it would be all-missing
# and is dropped rather than saved as a misleading all-NaN column.
df['female_x_children_x_ai_adoption'] = df['female'] * df['children_household'] * df['ai_adoption_enterprises_pct']  # H3 / H5
if df['female_x_children_x_ai_adoption'].notna().sum() == 0:
    df = df.drop(columns=['female_x_children_x_ai_adoption'])
    log_step('interactions', 'Built 5 required interaction variables. Optional female_x_children_x_ai_adoption dropped: AI adoption not available, would be all-missing')
else:
    log_step('interactions', 'Built 5 required interaction variables plus the optional female_x_children_x_ai_adoption')

interactions: Built 5 required interaction variables plus the optional female_x_children_x_ai_adoption


## 9. Final Missing Value and Quality Checks

In [18]:
final_columns = [
    'employed', 'age', 'age_squared', 'female', 'education_years', 'education_level',
    'high_education', 'married', 'children_household', 'household_size',
    'migration_background', 'health_status', 'digital_readiness',
    'institutional_trust_index', 'discrimination_experience', 'country',
    'ai_adoption_enterprises_pct', 'survey_weight',
    'education_years_x_ai_adoption', 'digital_readiness_x_ai_adoption',
    'female_x_children', 'migration_x_education', 'migration_x_digital_readiness',
]
if 'female_x_children_x_ai_adoption' in df.columns:
    final_columns.append('female_x_children_x_ai_adoption')

# Include any matched macro columns from section 7, if they were added.
final_columns += [c for c in df.columns if c not in final_columns and any(k in c for k in MACRO_KEYWORDS)] if df_macro_raw is not None else []
final_columns = [c for c in final_columns if c in df.columns]

final_df = df[final_columns].copy()

print('Final dataset shape:', final_df.shape)
print('\nFinal column list:')
print(list(final_df.columns))

Final dataset shape: (49837, 1291)

Final column list:
['employed', 'age', 'age_squared', 'female', 'education_years', 'education_level', 'high_education', 'married', 'children_household', 'household_size', 'migration_background', 'health_status', 'digital_readiness', 'institutional_trust_index', 'discrimination_experience', 'country', 'ai_adoption_enterprises_pct', 'survey_weight', 'education_years_x_ai_adoption', 'digital_readiness_x_ai_adoption', 'female_x_children', 'migration_x_education', 'migration_x_digital_readiness', 'female_x_children_x_ai_adoption', 'educde2', 'educgb1', 'c_cnmigrat_2000', 'c_cnmigrat_2001', 'c_cnmigrat_2002', 'c_cnmigrat_2003', 'c_cnmigrat_2004', 'c_cnmigrat_2005', 'c_cnmigrat_2006', 'c_cnmigrat_2007', 'c_cnmigrat_2008', 'c_cnmigrat_2009', 'c_cnmigrat_2010', 'c_cnmigrat_2011', 'c_cnmigrat_2012', 'c_cnmigrat_2013', 'c_cnmigrat_2014', 'c_cnmigrat_2015', 'c_cnmigrat_2016', 'c_cnmigrat_2017', 'c_cnmigrat_2018', 'c_cnmigrat_2019', 'c_cnmigrat_2020', 'c_cnmigrat

In [19]:
final_missing = (final_df.isna().mean() * 100).sort_values(ascending=False)
final_missing = final_missing[final_missing > 0]
print('Final missing values (percent), columns with at least one missing value:')
final_missing

Final missing values (percent), columns with at least one missing value:


reg11_soexgdp_2022               98.210165
reg11_soexgdp_2021               98.210165
reg11_soexgdp_2020               98.210165
reg11_soexgdp_2019               98.210165
reg11_soexgdp_1998               98.210165
                                   ...    
household_size                    0.609989
institutional_trust_index         0.307001
health_status                     0.136445
digital_readiness                 0.098321
migration_x_digital_readiness     0.098321
Length: 1281, dtype: float64

In [20]:
employed_distribution = final_df['employed'].value_counts(normalize=True)
print('Final employed distribution:')
print(employed_distribution)

country_coverage = final_df['country'].value_counts().sort_index()
print('\nFinal country coverage (respondent count per country):')
print(country_coverage)

Final employed distribution:
employed
1    0.51522
0    0.48478
Name: proportion, dtype: float64

Final country coverage (respondent count per country):
country
AT    2352
BE    1588
BG    2237
CH    1382
CY     677
DE    2412
EE    1290
ES    1841
FI    1558
FR    1765
GB    1668
GR    2754
HR    1556
HU    2116
IE    2013
IL     892
IS     835
IT    2846
LT    1355
LV    1242
ME    1549
NL    1682
NO    1334
PL    1431
PT    1372
RS    1550
SE    1226
SI    1243
SK    1429
UA    2642
Name: count, dtype: int64


In [21]:
variable_summary = final_df.describe(include='all').T
variable_summary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
employed,49837.0,NaN,NaN,NaN,0.51522,0.499773,0.0,0.0,1.0,1.0,1.0
age,49476.0,NaN,NaN,NaN,51.628567,18.654069,15.0,37.0,52.0,67.0,90.0
age_squared,49476.0,NaN,NaN,NaN,3013.476211,1931.767016,225.0,1369.0,2704.0,4489.0,8100.0
female,49837.0,NaN,NaN,NaN,0.539238,0.498463,0.0,0.0,1.0,1.0,1.0
education_years,49037.0,NaN,NaN,NaN,13.251178,4.015193,0.0,11.0,13.0,16.0,69.0
...,...,...,...,...,...,...,...,...,...,...,...
reg11_soexgdp_2019,892.0,NaN,NaN,NaN,16.068,0.0,16.068,16.068,16.068,16.068,16.068
reg11_soexgdp_2020,892.0,NaN,NaN,NaN,20.011,0.0,20.011,20.011,20.011,20.011,20.011
reg11_soexgdp_2021,892.0,NaN,NaN,NaN,17.771,0.0,17.771,17.771,17.771,17.771,17.771
reg11_soexgdp_2022,892.0,NaN,NaN,NaN,15.608,0.0,15.608,15.608,15.608,15.608,15.608


## 10. Save Final Dataset

In [22]:
csv_path = PROCESSED_DIR / 'final_employment_ai_dataset.csv'
pkl_path = PICKLE_DIR / 'final_employment_ai_dataset.pkl'
final_df.to_csv(csv_path, index=False)
final_df.to_pickle(pkl_path)

log_step('save_final_dataset', f'Saved final dataset with shape {final_df.shape} to {csv_path.name} and {pkl_path.name}')

save_final_dataset: Saved final dataset with shape (49837, 1291) to final_employment_ai_dataset.csv and final_employment_ai_dataset.pkl


## 11. Save Variable Dictionary and Data Engineering Report

In [23]:
final_summary_path = TABLES_DIR / 'final_dataset_summary.xlsx'
with pd.ExcelWriter(final_summary_path) as writer:
    variable_summary.reset_index().rename(columns={'index': 'variable'}).to_excel(writer, sheet_name='variable_summary', index=False)
    final_missing.reset_index().rename(columns={'index': 'variable', 0: 'missing_pct'}).to_excel(writer, sheet_name='missing_values', index=False)
    employed_distribution.reset_index().rename(columns={'index': 'employed', 'employed': 'share'}).to_excel(writer, sheet_name='employed_distribution', index=False)
    country_coverage.reset_index().rename(columns={'index': 'country', 'country': 'n_respondents'}).to_excel(writer, sheet_name='country_coverage', index=False)

print('Saved final dataset summary to:', final_summary_path)

Saved final dataset summary to: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\outputs\tables\final_dataset_summary.xlsx


In [24]:
variable_dictionary_md = f"""# Variable Dictionary

This dictionary lists every variable in `data/processed/final_employment_ai_dataset.csv`
/ `data/pickle/final_employment_ai_dataset.pkl`, produced by
`notebooks/03_data_engineering_and_final_dataset.ipynb`. This file is generated by
that notebook — update the notebook and re-run it rather than editing this file by
hand, so the two never drift out of sync.

## Dependent Variable

| Variable | Definition | Source | Coding rule | Unit | Missing values (post-cleaning) |
|---|---|---|---|---|---|
| `employed` | Whether the respondent is currently employed | ESS `{COL_EMPLOYMENT_STATUS}` (main activity, last 7 days) | 1 = employed (code {COL_EMPLOYED_CODE}), 0 = not employed | binary | rows with missing status were dropped (see data engineering report) |

## Individual-Level Independent Variables

| Variable | Definition | Source | Coding rule | Unit | Missing (%) |
|---|---|---|---|---|---|
| `age` | Respondent age | ESS `{COL_AGE}` | as reported | years | {final_missing.get('age', 0):.1f} |
| `age_squared` | Age squared (non-linear age effect) | derived from `age` | `age ** 2` | years² | {final_missing.get('age_squared', 0):.1f} |
| `female` | Gender indicator | ESS `{COL_GENDER}` | 1 = female (code {COL_GENDER_FEMALE_CODE}), 0 = male | binary | {final_missing.get('female', 0):.1f} |
| `education_years` | Years of full-time education completed | ESS `{COL_EDUCATION_YEARS}` | as reported | years | {final_missing.get('education_years', 0):.1f} |
| `education_level` | ISCED education level | ESS `{COL_EDUCATION_LEVEL}` | as reported | categorical | {final_missing.get('education_level', 0):.1f} |
| `high_education` | Tertiary education indicator | derived from `education_level` | 1 = ISCED >= {COL_HIGH_EDUCATION_THRESHOLD}, 0 = otherwise | binary | {final_missing.get('high_education', 0):.1f} |
| `married` | Marital status indicator | ESS `{COL_MARITAL}` | 1 = married (code {COL_MARRIED_CODE}), 0 = otherwise | binary | {final_missing.get('married', 0):.1f} |
| `children_household` | Child under 16 living in the household | ESS household grid: birth years `yrbrn2`–`yrbrn13` + interview year from `{COL_INTERVIEW_DATE}` | 1 = at least one co-resident member aged under {CHILD_AGE_CUTOFF}, 0 = otherwise | binary | {final_missing.get('children_household', 0):.1f} |
| `household_size` | Number of people in household | ESS `{COL_HOUSEHOLD_SIZE}` | as reported | count | {final_missing.get('household_size', 0):.1f} |
| `migration_background` | Migration background indicator | ESS `{COL_MIGRATION}` | 1 = not born in country (code {COL_MIGRATION_CODE}), 0 = otherwise | binary | {final_missing.get('migration_background', 0):.1f} |
| `health_status` | Self-reported general health | ESS `{COL_HEALTH}` | as reported (ordinal) | ordinal | {final_missing.get('health_status', 0):.1f} |
| `digital_readiness` | Internet use frequency (individual-level digital readiness proxy) | ESS `{COL_DIGITAL}` | as reported (ordinal) | ordinal | {final_missing.get('digital_readiness', 0):.1f} |
| `institutional_trust_index` | Composite trust in institutions | ESS trust items: {available_trust_items} | row-wise mean, skipping missing items | index | {final_missing.get('institutional_trust_index', 0):.1f} |
| `discrimination_experience` | Self-reported discrimination | ESS `{COL_DISCRIMINATION}` | 1 = experienced discrimination (code {COL_DISCRIMINATION_CODE}), 0 = otherwise | binary | {final_missing.get('discrimination_experience', 0):.1f} |
| `country` | Country of residence | ESS `{COL_COUNTRY}` | ISO country code | categorical | {final_missing.get('country', 0):.1f} |
| `survey_weight` | ESS analysis weight | ESS `{COL_WEIGHT}` | as reported | weight | {final_missing.get('survey_weight', 0):.1f} |

## Country-Level Variables

| Variable | Definition | Source | Coding rule | Unit | Missing (%) |
|---|---|---|---|---|---|
| `ai_adoption_enterprises_pct` | Share of enterprises with at least 10 employees using at least one AI technology | Eurostat `isoc_eb_ai`, year {ai_year_used} | `freq=A`, `size_emp=GE10`, `nace_r2=C10-S951_X_K`, `indic_is=E_AI_TANY`, `unit=PC_ENT` | percent | {final_missing.get('ai_adoption_enterprises_pct', 0):.1f} |

Additional macro control columns, if matched from ESS Multilevel data, are listed
in the final dataset summary (`outputs/tables/final_dataset_summary.xlsx`) and were
only added where clearly available — no missing indicator was invented.

## Interaction Variables

| Variable | Definition | Components | Hypothesis link |
|---|---|---|---|
| `education_years_x_ai_adoption` | Education effect moderated by country AI adoption | `education_years * ai_adoption_enterprises_pct` | H2 |
| `digital_readiness_x_ai_adoption` | Digital readiness effect moderated by country AI adoption | `digital_readiness * ai_adoption_enterprises_pct` | H2 |
| `female_x_children` | Family constraint effect for women | `female * children_household` | H3 |
| `migration_x_education` | Education's role in reducing the migrant employment gap | `migration_background * education_years` | H4 |
| `migration_x_digital_readiness` | Digital readiness' role in reducing the migrant employment gap | `migration_background * digital_readiness` | H4 |
| `female_x_children_x_ai_adoption` (optional) | Whether AI adoption changes the family-constraint effect for women | `female * children_household * ai_adoption_enterprises_pct` | H3 / H5 |

## Notes and Documented Assumptions

- All variable names use `lower_snake_case` and must be used identically in
  notebooks 04–07 — never renamed downstream.
- `children_household` is a direct household-grid indicator (any co-resident member
  aged under {CHILD_AGE_CUTOFF}); `digital_readiness` is a documented proxy (internet-use
  frequency), not a direct measure of individual AI use — see the notes above.
- Original ESS variable codes are kept as staging columns only during notebook 03;
  the final saved dataset contains only the renamed analysis variables.
- This file is regenerated every time notebook 03 is run; do not hand-edit it.
"""

variable_dictionary_path = DOCS_DIR / 'variable_dictionary.md'
with open(variable_dictionary_path, 'w', encoding='utf-8') as f:
    f.write(variable_dictionary_md)

print('Saved variable dictionary to:', variable_dictionary_path)

Saved variable dictionary to: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\docs\variable_dictionary.md


In [25]:
report_path = REPORTS_DIR / 'data_engineering_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('# Data Engineering Report\n\n')
    f.write('Generated by notebooks/03_data_engineering_and_final_dataset.ipynb.\n\n')
    f.write('## Cleaning and Construction Steps\n\n')
    f.write('| Step | Detail |\n|---|---|\n')

    # If an upstream cell is rerun in the same kernel, report only its latest
    # Eurostat-cleaning record rather than retaining stale diagnostics.
    clean_ai_indices = [
        i for i, (step, _) in enumerate(cleaning_log) if step == 'clean_eurostat_ai'
    ]
    latest_clean_ai_index = clean_ai_indices[-1] if clean_ai_indices else None
    for i, (step, detail) in enumerate(cleaning_log):
        if step == 'clean_eurostat_ai' and i != latest_clean_ai_index:
            continue
        f.write(f'| {step} | {detail} |\n')

    duplicate_expansion_fixed = (
        n_rows_before_ai_merge
        == n_rows_after_ai_merge
        == n_rows_after_macro_merge
        == len(final_df)
    )
    f.write('\n## Merge Row Audit\n\n')
    f.write(f'- Original ESS row count: {len(df_ess_raw)}\n')
    f.write(f'- Row count after employment-status cleaning: {n_after_drop}\n')
    f.write(f'- Row count before Eurostat AI merge: {n_rows_before_ai_merge}\n')
    f.write(f'- Eurostat AI rows before filtering: {eurostat_rows_before_filtering}\n')
    f.write(f'- Eurostat AI rows after filtering: {eurostat_rows_after_filtering}\n')
    f.write(f'- Unique countries in filtered Eurostat AI table: {eurostat_unique_countries}\n')
    f.write(f'- Row count after Eurostat AI merge: {n_rows_after_ai_merge}\n')
    f.write(f'- Row count after ESS Multilevel merge: {n_rows_after_macro_merge}\n')
    f.write(f'- Final modelling dataset row count: {len(final_df)}\n')
    f.write(
        '- Duplicate respondent expansion fixed: '
        f'{"yes" if duplicate_expansion_fixed else "no"}\n'
    )
    f.write('- Eurostat merge validation: many_to_one\n')
    f.write('- ESS Multilevel merge validation: many_to_one\n')

    f.write('\n## Final Dataset\n\n')
    f.write(f'- Shape: {final_df.shape}\n')
    f.write(f'- Employed distribution: {employed_distribution.to_dict()}\n')
    f.write(f'- Countries covered: {final_df["country"].nunique()}\n')
    f.write(f'- Columns with remaining missing values: {final_missing.to_dict()}\n')

print('Saved data engineering report to:', report_path)
print('Next step: notebooks/04_baseline_econometric_models.ipynb')

Saved data engineering report to: C:\Users\khamidov.m\OneDrive - Procter and Gamble\Desktop\Project\Project\employment_ai_europe_thesis\outputs\reports\data_engineering_report.md
Next step: notebooks/04_baseline_econometric_models.ipynb
